# 14c · Un job vero: gruppo, run e il device che chiede

Qui si vede cosa vuol dire **"un job è in esecuzione"**. Il servizio **non spinge** niente al device: la **run** è una porta aperta. Ogni device compatibile del gruppo, **quando chiede** sulla sua connessione DPS, trova l'aggiornamento; chi non ha ancora chiesto resta *non applicato*.

```text
  tu:      gruppo --- job (gruppo + aggiornamento) --- run (Active)
  device:  DPS (MQTT + certificato) --> "ho produttore Contoso, modello RobotV1, installato: niente"
           <-- manifest firmato + indirizzo dei file     (solo se c'e' una run Active che lo riguarda)
           verifica firma, scarica, controlla hash, installa, prova, riporta Succeeded o Failed
```

Il **device** è un **processo** (`device\su-agent\su-agent.cs`, C#): lo scriviamo noi, dai sorgenti dell'SDK C di Microsoft, perché il client ufficiale esiste solo in C (preview) e non c'è per PowerShell o .NET. Il servizio è vero.

**Serve:** i notebook 14a e 14b eseguiti, e `device-da-aggiornare` registrato (02c). Il primo avvio dell'agente compila il programma (qualche secondo).
**Da dire.** "Il job non installa niente da solo. Dice 'questo aggiornamento è per questo gruppo'. È il device che passa a prenderlo."


In [ ]:
# Carica la libreria e legge lo scenario (notebooks\_scenario.psd1): da qui in poi si usa $cfg
. ([scriptblock]::Create([IO.File]::ReadAllText((Get-ChildItem . , .\notebooks , .. -Filter _start.ps1 -ErrorAction SilentlyContinue | Select-Object -First 1).FullName)))

## 0 · Il device e i nomi
Il device, i suoi certificati di bootstrap (con cui parla con DPS) e dove tiene lo stato. La cella stampa anche l'**ID scope** del DPS.


In [ ]:
$id = 'device-da-aggiornare'                 # il device dell'aggiornamento (Contoso, RobotV1, software revision 1.0.0)
$provider = 'Contoso'; $nome = 'termostato'    # l'identita' dell'aggiornamento: provider / nome / versione
$ns = $adrName
$cartella = (Get-ChildItem (Join-Path $out "devices\$id") -Directory | Sort-Object Name | Select-Object -Last 1).FullName     # i certificati del device (notebook 02c)
$stato = Join-Path $out "su-agent\$id"                                       # dove il device tiene cio' che ha installato
$agente = (Resolve-Path (Join-Path $base '..\device\su-agent\su-agent.cs')).Path
$idScope = az iot dps show --name $dpsName --resource-group $rg --query properties.idScope -o tsv --only-show-errors
if (-not (Test-Path (Join-Path $cartella 'bootstrap-chain.pem'))) { throw "Il device $id non ha il certificato di bootstrap: esegui il notebook 02c." }
"device: $id | certificati: $cartella"
"id scope DPS: $idScope"


## 1 · Si parte pulito
Si cancellano i job, le run e i gruppi di una prova precedente, si riporta il device alla software revision `1.0.0` e si fa dimenticare al device quello che aveva installato.


In [ ]:
$gruppo = 'gruppo-robotv1-rev1'; $job = 'aggiorna-termostato-2'
foreach ($j in 'aggiorna-termostato-2', 'aggiorna-termostato-3') {
    foreach ($r in @(az iot adr ns job run list --job-name $j --ns $ns --resource-group $rg --query "[].name" -o tsv --only-show-errors 2>$null)) {
        az iot adr ns job run cancel --name $r --job-name $j --ns $ns --resource-group $rg --yes -o none --only-show-errors 2>$null
        az iot adr ns job run delete --name $r --job-name $j --ns $ns --resource-group $rg --yes -o none --only-show-errors 2>$null
    }
    az iot adr ns job delete --name $j --ns $ns --resource-group $rg --yes -o none --only-show-errors 2>$null
}
foreach ($g in 'gruppo-robotv1-rev1', 'gruppo-robotv1-rev2') { Remove-AdrGroup -Name $g -Namespace $ns -ResourceGroup $rg }
Set-AdrDeviceProperties -Name $id -Namespace $ns -ResourceGroup $rg -SoftwareRevision '1.0.0'
if (Test-Path $stato) { Remove-Item $stato -Recurse -Force }
"Pulito. $id e' alla software revision 1.0.0 e non ha installato niente."


## 2 · Un gruppo con una query
La query sulle proprietà in ADR: produttore, modello e software revision. Sono i device che **dovrebbero** ricevere l'aggiornamento. Non decide la compatibilità: quella la decide il manifest.


In [ ]:
$errore = New-AdrGroup -Name $gruppo -Query "WHERE manufacturer = 'Contoso' AND model = 'RobotV1' AND softwareRevision = '1.0.0'" -Namespace $ns -ResourceGroup $rg
if ($errore) { throw "Gruppo rifiutato: $errore" }
"Membri di ${gruppo}: $((Get-AdrGroupMembers -Name $gruppo -Namespace $ns -ResourceGroup $rg) -join ', ')"


## 3 · Il job
Il job è la **definizione**: *questo aggiornamento* (provider, nome, versione) *a questo gruppo*. Dopo la creazione **non si cambia** né il gruppo né l'aggiornamento. Un job non fa niente finché non c'è una run.


In [ ]:
az iot adr ns job create `
    --name $job `
    --ns $ns `
    --resource-group $rg `
    --type SoftwareUpdate `
    --target-group-name $gruppo `
    --update-id-provider $provider `
    --update-id-name $nome `
    --update-id-version 2.0.0 `
    --description "Termostato 2.0.0 ai robot V1" `
    --query "{job:name, tipo:properties.jobType, gruppo:properties.target.resourceId, aggiornamento:properties.definition.updateResourceId, programmazione:properties.definition.schedulingType}" `
    -o json `
    --only-show-errors
Assert-Az "Creazione del job"


## 4 · La run: se nessun device ha mai chiesto, fallisce subito
`job schedule` crea una **run**. Prima di partire il servizio controlla che **almeno un device del gruppo abbia dichiarato** (chiedendo) le proprietà di compatibilità. Se nessun device ha mai chiesto, la run finisce subito in `Failed` con `UpdateNotCompatibleWithGroup`:
*"You may need to wait for device agents to check for software updates and for these attributes to be populated before creating a new Job Run."* (visto così il 2026-11-22.)
La cella programma la run; se fallisce per questo motivo, **il device chiede una volta** (dichiara produttore e modello) e la run si riprogramma.


In [ ]:
function New-JobRun([string]$Run) {
    az iot adr ns job schedule --name $job --ns $ns --resource-group $rg --run-name $Run -o none --only-show-errors
    Assert-Az "Programmazione della run $Run"
    for ($i = 0; $i -lt 12; $i++) {
        Start-Sleep -Seconds 5
        $r = az iot adr ns job run show --name $Run --job-name $job --ns $ns --resource-group $rg --query "{stato:properties.status, errore:properties.error.code}" -o json --only-show-errors | ConvertFrom-Json
        if ($r.stato -ne 'Scheduled') { return $r }
    }
    $r
}
$r = New-JobRun 'run-1'
"run-1: $($r.stato) $($r.errore)"
if ($r.errore -eq 'UpdateNotCompatibleWithGroup') {
    "Nessun device ha ancora dichiarato la compatibilita': il device chiede una volta, poi si riprogramma la run."
    Invoke-DeviceUpdateCheck -Device $id -Folder $cartella -State $stato -IdScope $idScope -AgentFile $agente
    az iot adr ns job run delete --name run-1 --job-name $job --ns $ns --resource-group $rg --yes -o none --only-show-errors
    $r = New-JobRun 'run-2'
    "run-2: $($r.stato) $($r.errore)"
}


## 5 · La run è Active: cosa vede il servizio
`Active` vuol dire: **la porta è aperta**. Il riepilogo conta i device del gruppo: nessuno ha ancora chiesto, quindi sono tutti *non applicati*. Il risultato per device è ancora vuoto.


In [ ]:
$run = az iot adr ns job run list --job-name $job --ns $ns --resource-group $rg --query "[0].name" -o tsv --only-show-errors
"run: $run"
az iot adr ns job run summary --name $run --job-name $job --ns $ns --resource-group $rg -o json --only-show-errors
"risultati per device:"
az iot adr ns job run results --name $run --job-name $job --ns $ns --resource-group $rg -o json --only-show-errors


## 6 · Il device chiede: il processo
Il device si collega a DPS (MQTT su TLS, con il suo certificato di bootstrap), **chiede** "c'è un aggiornamento per me?" dichiarando produttore, modello e cosa ha installato. Il servizio risponde con il **manifest firmato** e l'indirizzo dei file.
Il device **verifica la firma** (chiave radice → chiave di firma → hash del manifest), **scarica** il file in HTTPS, controlla **dimensione e hash**, tiene da parte la versione buona, **installa**, **prova**, e **riporta** il risultato. Ogni riga è un passo.


In [ ]:
Invoke-DeviceUpdateCheck -Device $id -Folder $cartella -State $stato -IdScope $idScope -AgentFile $agente


## 7 · Cosa vede il servizio adesso
Il device ha riportato `SUCCEEDED` con il codice **700**. Il riepilogo: **1 riuscito**, **2 non applicati** (`robot-01` e `robot-02` sono nel gruppo ma **non hanno chiesto**: la run li aspetta). La run **resta Active**: non finisce da sola. Attenzione: il **riepilogo si aggiorna qualche decina di secondi dopo** il risultato per device (subito dopo il report mostra ancora 3 non applicati, mentre l'elenco per device è già giusto): la cella aspetta 30 secondi.


In [ ]:
Start-Sleep -Seconds 30      # il riepilogo si aggiorna qualche decina di secondi DOPO il risultato per device
az iot adr ns job run summary --name $run --job-name $job --ns $ns --resource-group $rg -o json --only-show-errors
az iot adr ns job run results --name $run --job-name $job --ns $ns --resource-group $rg -o json --only-show-errors | ConvertFrom-Json |
    ForEach-Object { [pscustomobject]@{ device = ($_.resourceId -split '/')[-1]; stato = $_.status; risultato = $_.installResult.outcome; codice = $_.installResult.resultCode } } | Format-Table | Out-String
"run: " + (az iot adr ns job run show --name $run --job-name $job --ns $ns --resource-group $rg --query properties.status -o tsv --only-show-errors)


## 8 · Il device cambia la software revision
Il device ha installato la 2.0.0, ma **ADR non lo sa**: la software revision è una proprietà che qualcuno deve aggiornare. Lo facciamo qui (come nel notebook 15). Il gruppo `$gruppo`, una fotografia, non se ne accorge finché non si rinfresca.


In [ ]:
Set-AdrDeviceProperties -Name $id -Namespace $ns -ResourceGroup $rg -SoftwareRevision '2.0.0'
az iot adr ns device show --name $id --namespace $ns --resource-group $rg --query "{device:name, softwareRevision:properties.softwareRevision}" -o table --only-show-errors
"Il device ha installato: $(Get-Content (Join-Path $stato 'installed.json') -Raw)"


## 9 · Chiudere la run
Una run **non finisce**: resta Active e serve i ritardatari. Si **chiude** con `run cancel`. Il device che ha già installato resta alla 2.0.0; chi non aveva chiesto non riceve più niente da questa run.


In [ ]:
az iot adr ns job run cancel --name $run --job-name $job --ns $ns --resource-group $rg --yes -o none --only-show-errors
Assert-Az "Chiusura della run"
"run: " + (az iot adr ns job run show --name $run --job-name $job --ns $ns --resource-group $rg --query properties.status -o tsv --only-show-errors)


## Cosa abbiamo visto
- Un **job** è una definizione (aggiornamento + gruppo, immutabile); la **run** è l'esecuzione e **resta Active** finché non la chiudi.
- Il servizio **non spinge**: il device **chiede** su DPS, con il suo certificato. Chi non chiede resta *non applicato*.
- Prima della run **almeno un device deve aver dichiarato** produttore e modello chiedendo: altrimenti `UpdateNotCompatibleWithGroup`.
- Il device **verifica la firma** del manifest e l'hash dei file prima di installare, e **riporta** il risultato (`SUCCEEDED`, codice 700) che il servizio mostra per device.
- La software revision in ADR **non cambia da sola**.
- Il device qui è un **nostro programma** che parla il protocollo dell'SDK C: non è il client Microsoft.

**Prossimo notebook: 14d, quando l'installazione fallisce.**
